# Fieldwise Cloud feature engineering analysis

## 1. Executive overview

Phase 05 turns historical subscription evidence into variables that can be inspected at the time of a possible prediction. The unit is **one canonical subscription episode at one UTC observation timestamp**. An account may have several episodes. Its identifier supports attribution and grouping, while each episode retains its own identity.

This notebook uses the accepted, fictional Fieldwise Cloud reference. Its purpose is to explain the feature system and the variation and omissions it produces. Population proportions, the catalogue and the event calendar describe an engineering scenario for Great Britain. They do not estimate GB market behaviour.

The candidate, prediction-eligible and resolved denominators answer different questions. The next sections recompute each population from the bundle and identify which denominator applies. Predictor values, account and observation metadata, prediction eligibility and retrospective outcomes are separate artefacts.

In [ ]:
from collections import Counter
from decimal import Decimal
from hashlib import file_digest
from json import loads
from math import ceil, isfinite
from pathlib import Path
from statistics import mean, median, stdev

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
from IPython import get_ipython
from IPython.display import display

from nowa_customer_intelligence.data.synthetic.feature_matrix import (
    FEATURE_BUNDLE_VERSION,
    FEATURE_SPEC_VERSION,
    FEATURE_SPECS,
)

project = next(
    (
        p
        for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "pyproject.toml").is_file() and (p / "notebooks").is_dir()
    ),
    None,
)
if project is None:
    raise RuntimeError("Start from the repository root or a notebook subdirectory.")

get_ipython().run_line_magic("matplotlib", "inline")
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"axes.titleweight": "bold", "figure.figsize": (10, 4)})
blue, orange, green = "#23667d", "#db8351", "#657657"

## 2. Evidence flow and point-in-time governance

**Fieldwise source projections and internal facts → canonical attribution → observation-time filters → registry-defined calculations → quality checks → ordered feature bundle**

| Stage | Evidence and control |
|---|---|
| Sources | The five Contract v1.1 projections supply the customer, episode, invoice, monthly usage and snapshot inventory. Internal lifecycle, snapshot, usage and billing facts also record availability. |
| Candidate and risk set | At time *t*, canonical identity, usable historical state and enough history establish prediction eligibility. An effective termination already known at *t* excludes the episode. |
| Feature window | The accepted experiment uses a 90-day lookback. Source occurrence, effective date and availability are checked where applicable. Monthly usage must also cover a whole UTC calendar month. |
| Quality and exports | Versioned ordered predictors are checked for schema, valid values and alignment. Observation keys and evidence timestamps are stored separately. |
| Retrospective branch | Outcome adjudication separately uses a 30-day experimental horizon. A late fact can change a classification without changing the earlier risk set or predictors. |

A right-censored observation can have historical features while follow-up ends before the outcome horizon is complete. It remains in the prediction population. See the [contract](../../docs/04_data/customer_data_contract.md) and [Phase 05 specification](../../docs/07_project_management/implementation_plan_and_roadmap.md#18-phase-05-point-in-time-feature-engineering).

## 3. Reference matrix, provenance and population

**Question.** Do the local files match the accepted export format, and how many rows are available for prediction or retrospective binary analysis?

Load the files produced by the approved Phase 05 export command. The notebook opens the bundle read-only and checks the manifest version, ordered schema, sizes and SHA-256 values before parsing the tables. If the bundle is absent, follow the [documented local replay procedure](../README.md#phase-05-feature-engineering). The notebook does not open the Registry or regenerate input.

In [ ]:
bundle = project / "data" / "processed" / "phase05-fieldwise-final"
if not (bundle / "manifest.json").is_file():
    raise FileNotFoundError(
        "Prepare the accepted Phase 05 local bundle as described in notebooks/README.md."
    )
manifest = loads((bundle / "manifest.json").read_text(encoding="utf-8"))
names = tuple(spec.name for spec in FEATURE_SPECS)
assert manifest["bundle_version"] == FEATURE_BUNDLE_VERSION
assert manifest["feature_spec_version"] == FEATURE_SPEC_VERSION
assert tuple(manifest["predictor_columns"]) == names
expected_files = {
    "observations.jsonl",
    "predictors.jsonl",
    "prediction_eligibility.jsonl",
    "outcomes.jsonl",
}
assert set(manifest["files"]) == expected_files


def read_verified(name):
    path = bundle / name
    expected = manifest["files"][name]
    with path.open("rb") as stream:
        assert file_digest(stream, "sha256").hexdigest() == expected["sha256"], name
    assert path.stat().st_size == expected["bytes"], name
    with path.open(encoding="utf-8") as stream:
        return [loads(line) for line in stream]


observations = read_verified("observations.jsonl")
prediction_eligibility = read_verified("prediction_eligibility.jsonl")
outcomes = read_verified("outcomes.jsonl")
predictor_records = read_verified("predictors.jsonl")
assert all(tuple(record) == names for record in predictor_records)


def observation_key(record):
    key = record["key"]
    return key["subscription_id"], pd.Timestamp(key["observed_at"])


keys = [observation_key(row) for row in observations]
candidate_keys = [observation_key(row) for row in prediction_eligibility]
eligible_keys = [observation_key(row) for row in prediction_eligibility if row["exclusion"] is None]
assert keys == eligible_keys and len(keys) == len(predictor_records)
assert candidate_keys == [observation_key(row) for row in outcomes]
assert candidate_keys == sorted(candidate_keys, key=lambda key: (key[1], key[0]))
assert len(set(candidate_keys)) == len(candidate_keys)
assert all(
    pd.Timestamp(row["latest_evidence_at"]) <= key[1]
    for row, key in zip(observations, keys, strict=True)
)
assert all(key[1].utcoffset() == pd.Timedelta(0) for key in candidate_keys)

# Use the registry to validate decoded types and retain monetary precision.
for record in predictor_records:
    for spec in FEATURE_SPECS:
        value = record[spec.name]
        if value is None:
            assert spec.nullable, spec.name
            continue
        if spec.dtype is Decimal:
            assert type(value) is str, spec.name
            value = record[spec.name] = Decimal(value)
        assert type(value) is spec.dtype, spec.name
        if spec.dtype is not str:
            assert isfinite(value) and (spec.signed or value >= 0), spec.name

features = pd.DataFrame(predictor_records, columns=names)
observed = pd.Series(pd.DatetimeIndex([key[1] for key in keys]), name="Observation time")
quarters = observed.dt.tz_localize(None).dt.to_period("Q").rename("Observation quarter")
types = Counter(spec.dtype.__name__ for spec in FEATURE_SPECS)
states = Counter(row["outcome"]["state"] for row in outcomes)
exclusions = Counter(row["exclusion"] for row in prediction_eligibility if row["exclusion"])
resolved = sum(row["retrospective_training_eligible"] for row in outcomes)
positive = sum(
    row["retrospective_training_eligible"] and row["binary_target"] == 1 for row in outcomes
)
negative = sum(
    row["retrospective_training_eligible"] and row["binary_target"] == 0 for row in outcomes
)
assert resolved == positive + negative
quality = manifest["quality"]
assert (len(features), len(names), len(candidate_keys), resolved) == (
    quality["rows"],
    quality["features"],
    quality["input_observations"],
    quality["retrospective_training_rows"],
)
assert dict(states) == quality["outcomes"] and dict(exclusions) == quality["prediction_exclusions"]
summary = pd.Series(
    {
        "Candidate observations": len(candidate_keys),
        "Prediction-eligible rows": len(features),
        "Predictor columns": len(names),
        "Resolved eligible outcomes": resolved,
        "Confirmed positives": positive,
        "Confirmed negatives": negative,
        "Right-censored candidates": states["right_censored"],
        "Excluded for insufficient history": exclusions["insufficient_observation_history"],
        "Observation metadata fields after flattening the key": sum(
            len(v) if k == "key" else 1 for k, v in observations[0].items()
        ),
        "Missing predictor cells": int(features.isna().sum().sum()),
        "Positive share of resolved outcomes (%)": round(100 * positive / resolved, 2),
    },
    name="Verified reference",
)
display(summary, pd.Series(types, name="Predictor columns by registry type"))
display(
    features.loc[
        :,
        [
            "episode_age_days",
            "billing_frequency",
            "monthly_recurring_gbp",
            "observed_usage_months",
            "latest_active_users",
            "active_users_change",
        ],
    ]
    .head(5)
    .reset_index(drop=True)
)

The sample displays only predictor fields and a display index. The four flattened metadata fields are canonical episode and observation time, account grouping and latest used evidence time. Labels and their future adjudication timestamps are never joined into the predictor DataFrame. Positive prevalence uses resolved eligible binary outcomes as its denominator. Coverage and missingness below use prediction-eligible rows unless a chart specifies another denominator.

## 4. Complete feature dictionary

**Question.** What does each predictor measure, which source field supports it and what does a null mean? Names, categories, calculations, types and nullability come from the ordered registry. The tables add the source, units and practical interpretation for each feature.

Every entry is measured per canonical episode-observation. The common restrictions are finite values, non-negative levels and counts, a coverage ratio in [0, 1], and catalogue-consistent plan, cadence and recurring GBP. Change features may be negative. The applicable evidence gates appear in each group table.

In [ ]:
# Supplement the registry with source fields, units and explanations.
field_context = {
    "episode_age_days": (
        "Lifecycle establishment effective_at",
        "days",
        "Episode tenure. Required.",
    ),
    "prior_completed_episodes": (
        "Canonical account ownership; prior establishment and termination facts",
        "episodes",
        "Known predecessors. Zero means none qualify.",
    ),
    "plan_id": ("Historical snapshot plan_id", "category", "Known commercial tier. Required."),
    "billing_frequency": (
        "Historical snapshot billing_frequency",
        "category",
        "Known charge cadence. Required.",
    ),
    "monthly_recurring_gbp": (
        "Historical snapshot recurring_revenue",
        "GBP/month",
        "Normalised basis. Required.",
    ),
    "issued_invoice_count": (
        "Invoice ID/date; linked billing fact",
        "invoices",
        "Available charges. Zero is a count.",
    ),
    "issued_amount_gbp": (
        "Invoice amount/date; linked billing fact",
        "GBP",
        "Issued total. Null if no qualifying charge.",
    ),
    "mean_invoice_gbp": (
        "Invoice amount/date; linked billing fact",
        "GBP/invoice",
        "Typical issue amount, rounded to pennies. Null if none qualify.",
    ),
    "days_since_last_invoice": (
        "Invoice issue date; linked billing fact",
        "days",
        "Issue recency. Null if no qualifying charge.",
    ),
    "expected_usage_months": (
        "Observation window and activation; whole UTC periods",
        "months",
        "Opportunity to observe. Zero means no whole month in scope.",
    ),
    "observed_usage_months": (
        "Monthly usage period bounds and available_at",
        "months",
        "Usable rows. Zero means none known.",
    ),
    "unobserved_usage_months": (
        "Expected and observed monthly evidence",
        "months",
        "As-known gap. Absence and lateness can look alike.",
    ),
    "usage_coverage_ratio": (
        "Expected and observed monthly evidence",
        "fraction",
        "Share of relevant months observed. Null for zero expected.",
    ),
    "active_user_months": (
        "Monthly usage active_users and available_at",
        "months",
        "Known active-user readings, including literal zero.",
    ),
    "zero_active_user_months": (
        "Monthly usage active_users",
        "months",
        "Known zero readings only.",
    ),
    "latest_active_users": (
        "Monthly usage active_users",
        "users",
        "Latest usable row; null if unknown there.",
    ),
    "mean_active_users": (
        "Monthly usage active_users",
        "users/month",
        "Known-value mean; null if none.",
    ),
    "max_active_users": (
        "Monthly usage active_users",
        "users",
        "Known-value maximum; null if none.",
    ),
    "latest_sessions": (
        "Monthly usage sessions_count",
        "sessions",
        "Latest usable row; null if unknown there.",
    ),
    "mean_sessions": (
        "Monthly usage sessions_count",
        "sessions/month",
        "Known-value mean; null if none.",
    ),
    "latest_events": (
        "Monthly usage events_count",
        "events",
        "Latest usable row; null if unknown there.",
    ),
    "mean_events": (
        "Monthly usage events_count",
        "events/month",
        "Known-value mean; null if none.",
    ),
    "active_users_change": (
        "Monthly usage active_users",
        "users",
        "Signed change; null without comparable months.",
    ),
    "sessions_change": (
        "Monthly usage sessions_count",
        "sessions",
        "Signed change; null without comparable months.",
    ),
    "events_change": (
        "Monthly usage events_count",
        "events",
        "Signed change; null without comparable months.",
    ),
}

groups = {
    "A. Episode and customer history": ("episode", "customer_history"),
    "B. Commercial": ("commercial",),
    "C. Billing": ("billing",),
    "D. Usage coverage": ("coverage",),
    "E. Usage levels": ("usage",),
    "F. Usage trends": ("trend",),
}
gates = {
    "A": "Establishment and any qualifying predecessor facts must be known at observation.",
    "B": "Use the latest usable snapshot. All four snapshot timestamps must meet the cutoff.",
    "C": "Issue date is in the lookback and linked billing evidence is known at observation.",
    "D": "Whole UTC months in scope from activation. Usage needs explicit availability.",
    "E": "The same completed-month and availability gates apply. No fills for absent values.",
    "F": "The same usage gates apply, with two consecutive rows and known values for the measure.",
}
assert tuple(field_context) == names
assert sorted(category for categories in groups.values() for category in categories) == sorted(
    {spec.category for spec in FEATURE_SPECS}
)
for title, categories in groups.items():
    print(title, "—", gates[title[0]])
    rows = []
    for spec in FEATURE_SPECS:
        if spec.category in categories:
            source_fields, unit, meaning = field_context[spec.name]
            rows.append(
                (
                    spec.name,
                    spec.category,
                    spec.dtype.__name__,
                    spec.nullable,
                    source_fields,
                    unit,
                    spec.calculation,
                    meaning,
                )
            )
    display(
        pd.DataFrame(
            rows,
            columns=[
                "Feature",
                "Category",
                "Type",
                "Nullable",
                "Source and fields",
                "Unit",
                "Registry calculation",
                "Meaning and missingness",
            ],
        ).set_index("Feature")
    )

### Reading the dictionary

A valid null can reflect an evidence boundary. An invoice sum is null if no charge qualifies. A trend is null when it cannot compare two consecutive months with known values. A zero in an observed active-user row is evidence of zero users for that row. A zero in a count of available rows only describes available evidence. None of these cases implies a failed payment.

## 5. Episode and customer history

**Question.** How much time and prior contractual history does the matrix represent? Episode age starts at that episode's establishment, even for returning accounts. The completed-episode count requires an earlier episode on the same account whose qualifying facts were available by this observation. A future successor does not contribute.

In [ ]:
episode_columns = ["episode_age_days", "prior_completed_episodes"]
display(features[episode_columns].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).round(2))
display(
    pd.crosstab(quarters, features["prior_completed_episodes"]).rename_axis(
        columns="Prior completed episodes"
    )
)
fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4), layout="constrained")
features["episode_age_days"].plot.hist(ax=left, bins=24, color=blue)
left.set(
    title="Fieldwise episode age at observation",
    xlabel="Days since establishment",
    ylabel="Prediction-eligible observations",
)
features["prior_completed_episodes"].value_counts().sort_index().plot.bar(
    ax=right, color=green, rot=0
)
right.set(
    title="Fieldwise known completed predecessor episodes",
    xlabel="Prior completed episodes",
    ylabel="Prediction-eligible observations",
)
plt.show()
print(f"The median episode age is {features['episode_age_days'].median():.0f} days.")

Account tenure and episode tenure answer different questions. Repeated snapshots give longer-lived episodes more representation in these row-weighted summaries. The count of known predecessors can encode the generator’s deterministic reactivation rule. Later modelling should check reach and transfer across independent episodes before treating the apparent patterns as useful.

## 6. Commercial evidence

**Question.** Which commercial fields vary in the reference, and which variations share the same source assumption? Plan and billing frequency are native categories. The monthly-normalised amount is a Decimal value from the latest usable snapshot.

In [ ]:
commercial = pd.crosstab(
    [features["plan_id"], features["billing_frequency"]], features["monthly_recurring_gbp"]
)
display(commercial.rename_axis(columns="Monthly-normalised GBP").replace(0, "—"))
for name in ("plan_id", "billing_frequency"):
    counts = features[name].value_counts(dropna=False)
    display(
        pd.DataFrame({"rows": counts, "share (%)": counts.div(len(features)).mul(100)}).round(2)
    )
fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
features["billing_frequency"].value_counts().sort_index().plot.bar(ax=left, rot=0, color=blue)
left.set(title="Fieldwise billing frequency", xlabel="Billing cadence", ylabel="Observation rows")
features["monthly_recurring_gbp"].value_counts().sort_index().plot.bar(ax=right, rot=0, color=green)
right.set(
    title="Fieldwise monthly-normalised recurring amount",
    xlabel="GBP per month",
    ylabel="Observation rows",
)
plt.show()

The verified reference has a single plan, so plan offers no within-reference variation. The crosstab shows how recurring amount and cadence are related under the fictional catalogue. Their association reflects the pricing rule. A later evaluation should assess plan and cadence in an appropriately varied population before assigning predictive meaning.

## 7. Issued billing and invoice recency

**Question.** How many charges can be attributed and confirmed available, and how does charge timing differ by cadence? Issued amount, mean issue amount and recency describe the subset with at least one qualifying invoice. The catalogue amount is monthly-normalised and comes from snapshots. Accounting recognition and cash collection are outside the source evidence.

The table uses exact Decimal monetary values for arithmetic and computes empirical nearest-rank percentiles on observed amounts. The chart converts GBP amounts to floating point for rendering only.

In [ ]:
def empirical_quantile(ordered, fraction):
    return ordered[max(0, ceil(len(ordered) * fraction) - 1)]


def describe_features(columns):
    rows = {}
    for name in columns:
        values = sorted(record[name] for record in predictor_records if record[name] is not None)
        row = {
            "count": len(values),
            "missing": len(features) - len(values),
            "zero": values.count(0),
        }
        if values:
            row.update(
                {
                    "mean": mean(values),
                    "median": median(values),
                    "standard deviation": stdev(values) if len(values) > 1 else None,
                    "minimum": values[0],
                    "p05": empirical_quantile(values, 0.05),
                    "p25": empirical_quantile(values, 0.25),
                    "p75": empirical_quantile(values, 0.75),
                    "p95": empirical_quantile(values, 0.95),
                    "maximum": values[-1],
                }
            )
        rows[name] = row
    return pd.DataFrame.from_dict(rows, orient="index")


def format_statistics(table):
    return table.map(lambda value: f"{value:,.2f}" if isinstance(value, Decimal) else value)


billing_columns = [
    "issued_invoice_count",
    "issued_amount_gbp",
    "mean_invoice_gbp",
    "days_since_last_invoice",
]
display(format_statistics(describe_features(billing_columns)))
billing_summary = pd.DataFrame(
    {
        cadence: {
            name: median(known) if (known := list(group[name].dropna())) else None
            for name in billing_columns
        }
        for cadence, group in features.groupby("billing_frequency")
    }
).rename_axis("Conditional median by feature")
display(format_statistics(billing_summary))
fig, axes = plt.subplots(2, 2, figsize=(13, 7), layout="constrained")
titles = [
    "Available invoice count",
    "Total issued amount",
    "Mean invoice amount",
    "Latest issue recency",
]
units = [
    "Invoices per observation",
    "Issued total (GBP)",
    "Mean per invoice (GBP)",
    "Days since issue",
]
for ax, name, title, unit in zip(axes.flat, billing_columns, titles, units, strict=True):
    values = features[name].dropna().astype(float)
    bins = range(-1, int(values.max()) + 2) if name == "issued_invoice_count" else 20
    if name == "issued_invoice_count":
        bins = [edge + 0.5 for edge in bins]
    values.plot.hist(ax=ax, bins=bins, color=blue, edgecolor="white")
    ax.set(title=f"Fieldwise {title.lower()}", xlabel=unit, ylabel="Observed rows")
plt.show()

A monthly cadence can produce several smaller invoices within the lookback. An annual cadence can produce no qualifying issue in the same interval. An issued-invoice count of zero describes the available evidence. It gives no basis to infer an unpaid invoice or zero accounting revenue. Missing recency and amounts are expected with that zero count.

## 8. Usage coverage and expected months

**Question.** How much of the eligible observation window contains available usage? A month counts towards the expected denominator only when it is a whole UTC month inside the lookback and starts on or after known activation. An in-progress month, or a partial month at a history boundary, is structurally outside this denominator. A qualifying month that has ended may still be unavailable because of reporting lag.

The coverage gap is the expected count minus available rows at the cutoff. A row that will arrive later and a completely absent row have the same as-known gap. A known zero is an available measurement and counts towards coverage.

In [ ]:
coverage_columns = [
    "expected_usage_months",
    "observed_usage_months",
    "unobserved_usage_months",
    "usage_coverage_ratio",
    "active_user_months",
]
coverage = features[coverage_columns]
assert (
    coverage["expected_usage_months"]
    .eq(coverage["observed_usage_months"] + coverage["unobserved_usage_months"])
    .all()
)
display(describe_features(coverage_columns))
display(coverage.groupby(coverage_columns, dropna=False).size().rename("rows"))
coverage_quarters = coverage.groupby(quarters).agg(
    rows=("usage_coverage_ratio", "size"),
    mean_row_ratio=("usage_coverage_ratio", "mean"),
    expected=("expected_usage_months", "sum"),
    observed=("observed_usage_months", "sum"),
    unobserved=("unobserved_usage_months", "sum"),
)
coverage_quarters["missing ratios"] = (
    coverage["usage_coverage_ratio"].isna().groupby(quarters).sum()
)
display(coverage_quarters.round(3))
fig, (left, right) = plt.subplots(1, 2, figsize=(14, 4), layout="constrained")
ratio_counts = (
    coverage["usage_coverage_ratio"].dropna().mul(100).round(2).value_counts().sort_index()
)
ratio_counts.plot.bar(ax=left, color=blue, rot=0)
left.set(
    title="Fieldwise usage coverage per observation",
    xlabel="Observed / expected months (%)",
    ylabel="Observation rows",
)
coverage_quarters[["observed", "unobserved"]].plot.bar(
    ax=right, stacked=True, color=[green, orange], rot=45
)
right.set(
    title="Fieldwise observed and unavailable month opportunities",
    xlabel="Observation quarter",
    ylabel="Episode-month opportunities across rows",
)
plt.show()

The coverage rates form a small number of discrete bands because the window contains only a few whole months. The stacked counts add opportunities across observations and therefore count some episode-months repeatedly. The last month at each snapshot is unavailable under the reference reporting lag. A different operational calendar or reporting delay could produce a different coverage distribution. Later analysis should validate availability by source and period before comparing engagement levels.

## 9. Observed usage levels and literal zeros

**Question.** How do most-recent levels and historical means differ, and which large values are unusual within the observed reference? Latest refers to the last available monthly row, even if its particular metric is null. A mean uses known values only. A zero-active-user-month value counts rows that explicitly recorded zero users.

In [ ]:
usage_columns = [spec.name for spec in FEATURE_SPECS if spec.category == "usage"]
display(describe_features(usage_columns).round(2))
pairs = [
    ("latest_active_users", "mean_active_users"),
    ("latest_sessions", "mean_sessions"),
    ("latest_events", "mean_events"),
]
usage_comparison = {}
fig, axes = plt.subplots(2, 3, figsize=(15, 7), layout="constrained")
for index, (latest, average) in enumerate(pairs):
    observed_pair = features[[latest, average]].dropna()
    usage_comparison[latest] = {
        "known pairs": len(observed_pair),
        "equal readings": observed_pair[latest].eq(observed_pair[average]).sum(),
        "latest zeros": observed_pair[latest].eq(0).sum(),
    }
    label = field_context[latest][1]
    values = observed_pair[latest].astype(int)
    width = {"users": 1, "sessions": 1, "events": 7}[label]
    edges = [edge - 0.5 for edge in range(0, int(values.max()) + width + 1, width)]
    ax = axes[0, index]
    values.plot.hist(ax=ax, bins=edges, color=blue, edgecolor="white")
    ax.set(
        title=f"Fieldwise latest {label}",
        xlabel=f"{label.capitalize()} per row (bin {width})",
        ylabel="Observation rows",
    )
    ax = axes[1, index]
    ax.boxplot([observed_pair[latest], observed_pair[average]], tick_labels=["Latest", "Mean"])
    ax.set(
        title=f"Fieldwise latest versus mean {label}",
        xlabel="Available measure",
        ylabel=f"{label.capitalize()} per usage month",
    )
display(pd.DataFrame(usage_comparison).T)
plt.show()

# Tukey fences identify unusually high readings for review, with no removal or alteration.
tail = []
for latest, _ in pairs:
    data = features[latest].dropna()
    q1, q3 = data.quantile([0.25, 0.75])
    upper = q3 + 1.5 * (q3 - q1)
    tail.append(
        {
            "feature": latest,
            "upper fence": upper,
            "maximum": data.max(),
            "rows above fence": data.gt(upper).sum(),
        }
    )
display(pd.DataFrame(tail).set_index("feature"))

Latest and mean often coincide because many rows have only one available month. The distribution of zero-user months and the upper-tail review describe observed records. Unusual values are retained and checked against the fictional bounded scenario. Co-movement across users, sessions and events could reflect the generator or the shared observation process. Inspect the relationships alongside available-month counts before drawing a behavioural interpretation.

## 10. Usage trends and coverage constraints

**Question.** When can the reference support a change calculation? Each trend is the latest value minus the preceding value for the same measure, using two comparable consecutive months available by observation. Zero means unchanged between those two readings. Null means there is insufficient comparable evidence.

In [ ]:
trend_columns = [spec.name for spec in FEATURE_SPECS if spec.category == "trend"]
trends = features[trend_columns]
trend_support = pd.DataFrame(
    {
        "Unavailable": trends.isna().sum(),
        "Negative": trends.lt(0).sum(),
        "Zero": trends.eq(0).sum(),
        "Positive": trends.gt(0).sum(),
    }
).reindex(trend_columns)
trend_support["Available"] = trend_support[["Negative", "Zero", "Positive"]].sum(axis=1)
trend_support["Unavailable (%)"] = trend_support["Unavailable"].div(len(features)).mul(100)
assert trend_support[["Unavailable", "Available"]].sum(axis=1).eq(len(features)).all()
display(trend_support.round(2))
dense_trends = trends.notna().all(axis=1).rename("All trends available")
display(
    pd.crosstab(
        [coverage["expected_usage_months"], coverage["observed_usage_months"]], dense_trends
    )
)
quarterly_trends = trends.notna().groupby(quarters).mean().mul(100)
display(quarterly_trends.round(2))
fig, (left, right) = plt.subplots(1, 2, figsize=(14, 4), layout="constrained")
trend_support[["Negative", "Zero", "Positive", "Unavailable"]].rename(
    index={
        "active_users_change": "Active users",
        "sessions_change": "Sessions",
        "events_change": "Events",
    }
).plot.barh(ax=left, stacked=True, color=[blue, "#9b9b9b", green, "#d7d7d7"])
left.set(
    title="Fieldwise observed direction and unavailable trends",
    xlabel="Prediction-eligible observations",
    ylabel="Usage measure",
)
complete_share = dense_trends.groupby(quarters).mean().mul(100)
complete_share.plot(ax=right, color=blue, marker="o")
right.set(
    title="Fieldwise complete trend support by quarter",
    xlabel="Observation quarter",
    ylabel="Rows with all three trends (%)",
    ylim=(0, 100),
)
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(13, 4), layout="constrained")
for ax, name in zip(axes, trend_columns, strict=True):
    unit = field_context[name][1]
    trends[name].dropna().plot.hist(ax=ax, bins=21, color=blue, edgecolor="white")
    ax.axvline(0, color=orange, linestyle="--")
    ax.set(
        title=f"Fieldwise change in {unit}",
        xlabel=f"Signed change ({unit})",
        ylabel="Rows with a trend",
    )
plt.show()

The coverage cross-tab shows which observation rows have enough available months. In this reference the missingness patterns of the three trends coincide. The signed distributions use the available subset only, so their balance cannot be imputed to the full prediction population. Future work should evaluate the consequences of systematically missing history before any modelling-specific handling.

## 11. Missing-data patterns

**Question.** Which values are absent together and does that pattern change by period? Compare observed missingness to the manifest, then describe the pattern at row and quarter levels. A null can be intentional when point-in-time evidence is unavailable. A missing mandatory field or an inconsistent relationship is a separate quality issue.

In [ ]:
missing = features.isna()
missing_summary = pd.DataFrame(
    {
        "category": [spec.category for spec in FEATURE_SPECS],
        "missing": missing.sum().reindex(names).to_numpy(),
    },
    index=pd.Index(names, name="Feature"),
)
missing_summary["missing (%)"] = missing_summary["missing"].div(len(features)).mul(100)
assert missing_summary["missing"].to_dict() == {
    name: evidence["missing"] for name, evidence in quality["inventory"].items()
}
display(missing_summary.sort_values("missing", ascending=False).round(2))
missing_groups = pd.DataFrame(
    {
        "Billing detail": missing[billing_columns].any(axis=1),
        "Usage trends": missing[trend_columns].any(axis=1),
    }
)
display(missing_groups.value_counts().rename("Observation rows"))
display(missing_groups.groupby(quarters).mean().mul(100).round(2))
fig, (left, right) = plt.subplots(1, 2, figsize=(15, 5), layout="constrained")
nonzero = missing_summary.loc[missing_summary["missing"].gt(0), "missing (%)"].sort_values()
nonzero.plot.barh(ax=left, color=orange)
left.set(
    title="Fieldwise predictors with observed missingness",
    xlabel="Missing rows (%)",
    ylabel="Registry feature",
    xlim=(0, 100),
)
missing_groups.groupby(quarters).mean().mul(100).plot(ax=right, marker="o")
right.set(
    title="Fieldwise missingness by evidence group",
    xlabel="Observation quarter",
    ylabel="Rows missing at least one value in group (%)",
    ylim=(0, 100),
)
plt.show()

The invoice fields are jointly null where there is no qualifying available issue. The available invoice count remains valid there. The trend fields are jointly null when a second usable month is absent. Complete columns can still be constant, strongly rule-bound or based on incomplete underlying history. These are reasons to inspect the process as well as the percentage. This analysis preserves all nulls.

## 12. Statistical distribution inventory

**Question.** What are the main shapes and edge values across the full feature registry? The summary includes observed counts, nulls, zeros, sample standard deviation, median and nearest-rank percentiles. Decimal arithmetic is retained for GBP measures. Different columns use different units, so ranges are interpreted alongside the feature dictionary.

In [ ]:
numeric_columns = [spec.name for spec in FEATURE_SPECS if spec.dtype is not str]
categorical_columns = [spec.name for spec in FEATURE_SPECS if spec.dtype is str]
statistical_summary = describe_features(numeric_columns)
display(format_statistics(statistical_summary.round(3)))
for name in categorical_columns:
    counts = features[name].value_counts(dropna=False)
    display(
        pd.DataFrame({"count": counts, "share (%)": counts.div(len(features)).mul(100)})
        .rename_axis(name)
        .round(2)
    )
variation = features.nunique(dropna=True).rename("Distinct observed values")
display(variation[variation.le(2)].sort_values())

Inspect low-variation features alongside the coverage table. For example, the as-known gap can be constant because the observation cadence and reporting delay are built into the reference scenario. The high share of zero predecessors reflects the generator and the snapshot mix. Counts and extrema measure the fictional evidence. They provide no estimate of predictive importance.

## 13. Relationships between predictors

**Question.** Which numerical relationships reflect shared evidence, accounting rules or cohort structure? Spearman correlation describes monotonic relationships using pairwise available observations. The heatmap omits constant columns because correlation is undefined for them. Associated pair support is reported separately. No labels, identifiers or timestamps enter either calculation.

In [ ]:
numeric = features[numeric_columns].astype(float)
varying = numeric.loc[:, numeric.nunique().gt(1)]
correlation = varying.corr(method="spearman", min_periods=2)
present = varying.notna().astype(int)
pair_support = present.T @ present
assert list(correlation.columns) == list(pair_support.columns)
upper = pair_support.where(
    pd.DataFrame(
        [[left < right for right in range(len(pair_support))] for left in range(len(pair_support))],
        index=pair_support.index,
        columns=pair_support.columns,
    )
)
print("Available pair counts range from", int(upper.min().min()), "to", int(upper.max().max()))
fig, ax = plt.subplots(figsize=(13, 10), layout="constrained")
mesh = ax.imshow(correlation, cmap="RdBu_r", vmin=-1, vmax=1)
labels = [name.replace("_", " ") for name in correlation.columns]
ax.set_xticks(range(len(labels)), labels=labels, rotation=60, ha="right", fontsize=8)
ax.set_yticks(range(len(labels)), labels=labels, fontsize=8)
ax.set(
    title="Fieldwise predictor-only Spearman correlation",
    xlabel="Numerical predictor",
    ylabel="Numerical predictor",
)
ax.grid(False)
fig.colorbar(mesh, ax=ax, label="Spearman rank correlation")
plt.show()

selected_pairs = [
    ("episode_age_days", "issued_amount_gbp"),
    ("episode_age_days", "latest_active_users"),
    ("latest_active_users", "latest_sessions"),
    ("latest_sessions", "latest_events"),
    ("issued_invoice_count", "issued_amount_gbp"),
    ("usage_coverage_ratio", "active_users_change"),
]
display(
    pd.DataFrame(
        [
            {
                "predictor A": x,
                "predictor B": y,
                "observed pairs": pair_support.loc[x, y],
                "Spearman": correlation.loc[x, y],
            }
            for x, y in selected_pairs
        ]
    ).round(3)
)
fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4), layout="constrained")
for ax, x, y, xlabel, ylabel in [
    (left, "episode_age_days", "issued_amount_gbp", "Episode age (days)", "Issued total (GBP)"),
    (right, "latest_sessions", "latest_events", "Latest sessions", "Latest events"),
]:
    pair = numeric[[x, y]].dropna()
    ax.scatter(pair[x], pair[y], s=7, alpha=0.12, color=blue, linewidths=0)
    ax.set(title=f"Fieldwise {xlabel.lower()} and {ylabel.lower()}", xlabel=xlabel, ylabel=ylabel)
plt.show()

Correlations within the coverage block can arise from arithmetic identities. Billing relationships can arise from cadence and catalogue prices. Latest and mean activity share observations, especially when only one month is visible. Correlations are row-weighted and pairs can have different support. A weak or undefined relationship on limited rows should be revisited with adequate evidence. These analyses provide no causal or predictive ranking.

## 14. Temporal integrity and leakage prevention

**Question.** How does an observation relate to the lookback and evidence cutoff? First select a verified reference row with two available usage months and at least one available invoice. Its metadata show the actual observation time and latest used evidence timestamp. Its predictors show included counts. The exported bundle contains aggregates rather than individual invoice and usage rows, so an illustrative timeline then explains item-level timing without claiming to reconstruct that row's source facts.

In [ ]:
experiment = manifest["configuration"]["experiment"]
lookback_days = experiment["history_days"]
horizon_days = experiment["horizon_days"]
example_index = features.index[
    coverage["observed_usage_months"].ge(2) & features["issued_invoice_count"].gt(0)
][0]
t = observed.iloc[example_index]
start = t - pd.Timedelta(days=lookback_days)
latest_used = pd.Timestamp(observations[example_index]["latest_evidence_at"])
horizon = t + pd.Timedelta(days=horizon_days)
display(
    pd.Series(
        {
            "Observation (UTC)": t,
            "Lookback begins (UTC)": start,
            "Latest used evidence (UTC)": latest_used,
            "Retrospective horizon ends (UTC)": horizon,
        },
        name="Verified timing, identifiers omitted",
    )
)
display(
    features.loc[
        example_index,
        [
            "episode_age_days",
            "issued_invoice_count",
            "issued_amount_gbp",
            "days_since_last_invoice",
            "expected_usage_months",
            "observed_usage_months",
            "unobserved_usage_months",
            "latest_active_users",
            "active_users_change",
        ],
    ].rename("Values from the same verified row")
)

# The positions below are illustrative evidence dates anchored on the real observation.
illustrative = [
    ("Episode established", start - pd.Timedelta(days=18), blue),
    ("Available invoice fact", t - pd.Timedelta(days=44), green),
    ("Completed month reported", t.normalize().replace(day=1), green),
    ("Month ending at observation", t, orange),
    ("That month reported after cutoff", t.normalize() + pd.Timedelta(days=1), orange),
    ("Future outcome horizon", horizon, "#7e6b93"),
]
fig, ax = plt.subplots(figsize=(12, 4.5), layout="constrained")
ax.axvspan(start, t, color=blue, alpha=0.12, label=f"{lookback_days}-day experimental lookback")
ax.axvline(t, color="#343434", linestyle="--", label="Verified observation cutoff")
for y, (_, when, colour) in enumerate(illustrative):
    ax.scatter(when, y, color=colour, s=48)
ax.set_yticks(range(len(illustrative)), labels=[label for label, _, _ in illustrative])
ax.annotate("At cutoff", (t, 3), xytext=(7, 8), textcoords="offset points")
ax.annotate(
    "Next-day 00:00, after cutoff",
    (illustrative[4][1], 4),
    xytext=(7, 8),
    textcoords="offset points",
)
ax.invert_yaxis()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
ax.set(
    title="Fieldwise illustrative evidence timeline anchored on a verified cutoff",
    xlabel="Calendar date (UTC)",
    ylabel="Illustrative evidence event",
)
ax.legend(loc="upper right")
plt.show()

Only a completed usage month with explicit available evidence can contribute. It can end at the observation cutoff and still be excluded if it is reported the following day. Likewise an issue date in the window is insufficient if linked billing evidence becomes available later. The 30-day outcome horizon governs adjudication and does not extend predictor availability.

The displayed row verifies alignment and the latest evidence timestamp for the exported reference. The illustrative positions explain the rule. Executable [feature tests](../../tests/unit/data/test_phase05_features.py) cover observation boundaries, late billing and usage, and future termination and reactivation. The [matrix integration test](../../tests/integration/test_phase05_feature_matrix.py) covers label isolation, the prediction roster and deterministic files. A timeline alone provides no leakage proof.

## 15. Feature quality dashboard

**Question.** Which quality claims can be checked directly from this export? Recompute key uniqueness, type consistency, alignment and missingness. Reconcile these with the manifest from the accepted run. The manifest also reports the upstream feature validity checks and source projection evidence.

In [ ]:
def aligned_allowlist():
    return (
        set(features.columns).isdisjoint(
            {
                "subscription_id",
                "customer_id",
                "observed_at",
                "latest_evidence_at",
                "binary_target",
                "horizon_end",
                "ascertainment_cutoff",
                "outcome",
            }
        )
        and tuple(features.columns) == names
    )


assert aligned_allowlist()
assert len(keys) - len(set(keys)) == quality["duplicate_keys"] == 0
assert quality["invalid_values"] == 0
assert coverage["usage_coverage_ratio"].dropna().between(0, 1).all()
assert all(record["status"] == "passed" for record in quality["inventory"].values())
dashboard = pd.Series(
    {
        "Eligible observation rows": len(keys),
        "Predictor columns": features.shape[1],
        "Unique observation keys": len(set(keys)),
        "Duplicate eligible keys": len(keys) - len(set(keys)),
        "Invalid values reported by exporter": quality["invalid_values"],
        "Predictors with missing values": int(missing.any().sum()),
        "Constant observed features": int(variation.eq(1).sum()),
        "Minimum usage coverage (%)": round(100 * coverage["usage_coverage_ratio"].min(), 2),
        "Maximum usage coverage (%)": round(100 * coverage["usage_coverage_ratio"].max(), 2),
        "Verified bundle data files": len(expected_files),
    },
    name="Feature quality",
)
display(dashboard)
display(variation[variation.eq(1)])
source_inventory = pd.DataFrame(manifest["inputs"])
display(source_inventory[["dataset", "records", "present", "sha256_hex"]])

The notebook directly checked the exported file hashes, ordered schema, decoded value types, metadata time alignment, key uniqueness and feature summaries. The source hashes and upstream invalid-value count are evidence recorded by the accepted export. The temporary projections from that run are outside this notebook's inputs. Same-Registry byte replay was verified during Phase 05 acceptance. This notebook verifies the retained bundle and performs no replay. Independent Registry identities and production evidence-availability policies remain outside this evidence.

## 16. Modelling readiness and conclusions

The matrix provides a clean episode-time predictor namespace and verifiable historical boundaries. Age and available episode history describe the lifecycle. Invoice count, issued value and recency describe available billing evidence. Usage levels describe observed months, and trend coverage is materially thinner. Consider each group in the context of its evidence gate before drawing business conclusions.

The single-plan catalogue limits within-reference comparison. The one-day reactivation schedule, temporally concentrated churn and severe class imbalance are generator-specific constraints. Incomplete monthly history and high trend missingness constrain coverage, while billed amounts rely on fictional issue-availability evidence. These patterns give useful engineering tests, with no guarantee of predictive performance or GB representativeness.

Before a later modelling phase, agree on point-in-time evidence policies, treatment of structurally unavailable and missing inputs, and a temporal evaluation design with adequate independent positive support. Use a fresh independent holdout because the earlier scored partition is development evidence. Apply any future preprocessing inside the authorised training workflow. This notebook creates no partitions, evaluates no algorithms and fits no models.